# Attention Mechanism

## Housekeeping

In [2]:
import sys
import os
import subprocess

print("Python executable:", sys.executable)
print("Python prefix:", sys.prefix)
print("Conda environment:", os.environ.get('CONDA_DEFAULT_ENV'))
print("Conda prefix:", os.environ.get('CONDA_PREFIX'))

# Check pip location
result = subprocess.run(['which', 'pip'], capture_output=True, text=True)
print("Pip location:", result.stdout.strip())

# Check pip version
result = subprocess.run(['pip', '--version'], capture_output=True, text=True)
print("Pip version:", result.stdout.strip())

# Add environment's bin directory to PATH
env_bin = os.path.dirname(sys.executable)
current_path = os.environ['PATH']
if env_bin not in current_path:
    os.environ['PATH'] = f"{env_bin}:{current_path}"

# Verify fix
result = subprocess.run(['which', 'pip'], capture_output=True, text=True)
print("Fixed pip location:", result.stdout.strip())

Python executable: /home/sagemaker-user/.conda/envs/venv-LLMsFromScratch/bin/python3
Python prefix: /home/sagemaker-user/.conda/envs/venv-LLMsFromScratch
Conda environment: base
Conda prefix: /opt/conda
Pip location: /opt/conda/bin/pip
Pip version: pip 25.1.1 from /opt/conda/lib/python3.12/site-packages/pip (python 3.12)
Fixed pip location: /home/sagemaker-user/.conda/envs/venv-LLMsFromScratch/bin/pip


## Code

## Simple Attention

In [3]:
import torch

inputs = torch.tensor(
   [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

In [4]:
inputs

tensor([[0.4300, 0.1500, 0.8900],
        [0.5500, 0.8700, 0.6600],
        [0.5700, 0.8500, 0.6400],
        [0.2200, 0.5800, 0.3300],
        [0.7700, 0.2500, 0.1000],
        [0.0500, 0.8000, 0.5500]])

In [5]:
# K, Q, V

query = inputs[1] # Selecting the Journey Keyword - This serves as Q (Query) in [K, Q, V]

attn_scores_2 = torch.empty(inputs.shape[0]) # Creating an empty tensor to store the calculated context vector of word 'journey'

# attn_scores_2.shape # The shape is 6 because the context is to be calculated with six words

for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i, query)

print(attn_scores_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


## Understanding dot products

In [14]:
# Attention Weight of work Journey with respect to word your
inputs[0][0]*inputs[1][0] + inputs[0][1]*inputs[1][1] + inputs[0][2]*inputs[1][2] 

tensor(0.9544)

In [11]:
print(inputs.shape[0])

6


In [15]:
# In order to make sure we can use the context - or self attention for this word we need to normalize the values
attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()

print("Attention Weights for Second Word:" , attn_weights_2_tmp)
print("Sum of all attention weights" , attn_weights_2_tmp.sum())

Attention Weights for Second Word: tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Sum of all attention weights tensor(1.0000)


In [17]:
# Recommended is to use softmax because - This approach is better at managing extreme values and offers more favorable gradient properties during training. 

def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)

attn_weight_2_naive = softmax_naive(attn_scores_2)

print(attn_weight_2_naive)
print(attn_weight_2_naive.sum())

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
tensor(1.)


In [18]:
# Using pytorch because our softmax implementation can behave stupidityly

attn_weights_2 = torch.softmax(attn_scores_2, dim=0)
print(attn_weights_2)
print(attn_weights_2.sum())

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
tensor(1.)


In [19]:
attn_weights_2.shape

torch.Size([6])

In [20]:
inputs.shape

torch.Size([6, 3])

In [24]:
attn_weights_2[0]*inputs[0] + attn_weights_2[1]*inputs[1] + attn_weights_2[2]*inputs[2] + attn_weights_2[3]*inputs[3] + attn_weights_2[4]*inputs[4] + attn_weights_2[5]*inputs[5]

tensor([0.4419, 0.6515, 0.5683])

In [27]:
query = inputs[1]

contect_vec_2 = torch.zeros(query.shape)

for i, x_i in enumerate(inputs):
    contect_vec_2 += attn_weights_2[i]*x_i

print(contect_vec_2)

tensor([0.4419, 0.6515, 0.5683])


## Computing attention weights for all input tokens

In [16]:
# Calculated attention score for every token

attn_scores = torch.empty(6, 6)

for i, x_i in enumerate(inputs):
    for j,x_j in enumerate(inputs):
        attn_scores[i,j] = torch.dot(x_i, x_j)
print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


### Attention Scores

In [17]:
# alternative way of calculating attention score

attn_scores = inputs @ inputs.T
print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


### Attention Weights

In [18]:
# Normalized attention score

attn_weights = torch.softmax(attn_scores, dim=-1)
print(attn_weights)

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


In [31]:
row_sum = lambda tensor1, row: tensor1[row].sum()
result = row_sum(attn_scores, 2) 
result

tensor(6.4801)

In [32]:
row_sum = lambda tensor1, row: tensor1[row].sum()
result = row_sum(attn_weights, 2) 
result

tensor(1.0000)

### Context Vector

In [33]:
all_context_vecs = attn_weights @ inputs
print(all_context_vecs)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


## Implementing self-attention with trainable weights

### Computing the attention weights step by step

In [34]:
x_2 = inputs[1]  #1 The second input element 

d_in = inputs.shape[1] #2 The input embedding size, d=3 

d_out = 2 #3 The output embedding size, d_out=2 


In [37]:
# Initialize the weight matrices
torch.manual_seed(123)
W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

In [39]:
# calculating the q, k, v vectors for token Journey

query_2 = x_2 @ W_query
key_2 = x_2 @ W_key
value_2 = x_2 @ W_value
print(query_2)
print(key_2)
print(value_2)

tensor([0.4306, 1.4551])
tensor([0.4433, 1.1419])
tensor([0.3951, 1.0037])


### Now calculating for all tokens

In [43]:
querys = inputs @ W_query
keys = inputs @ W_key
values = inputs @ W_value

print("querys.shape:", querys.shape)
print("keys.shape:", keys.shape)
print("values.shape:", values.shape)

querys.shape: torch.Size([6, 2])
keys.shape: torch.Size([6, 2])
values.shape: torch.Size([6, 2])


In [45]:
attn_score_2 = query_2.dot(key_2)
attn_score_2

tensor(1.8524)

In [46]:
attn_scores_2 = query_2 @ keys.T
attn_scores_2

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])

In [47]:
d_k = keys.shape[-1]

attn_weights_2 = torch.softmax(attn_scores_2 / d_k**0.5, dim = -1)

print(attn_weights_2)

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])


In [52]:
attn_weights_2.sum()

tensor(1.)

In [56]:
context_vec_2 = attn_weights_2 @ values
context_vec_2

tensor([0.3061, 0.8210])

## Implementing a compact self-attention Python class

In [65]:
import torch.nn as nn

class SelfAttention_v1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out)) # nn.Parameter() - Makes tensors trainable - PyTorch will update these during backprop
        self.W_key = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self,x):
            queries = x @ self.W_query
            keys = x @ self.W_key
            values = x @ self.W_value
            attn_scores = queries @ keys.T
            attn_weights = torch.softmax( attn_scores / keys.shape[-1]**0.5, dim = -1)
            context_vec = attn_weights @ values
            return context_vec

In [66]:
torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in,d_out)
print(sa_v1(inputs))

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


# ----------------------------------------------------------------------

In [1]:
import torch